# 05 — Single final test session (§0.7 — run ONCE)

**The test set is evaluated once, at the end, with the val-selected checkpoint
of each of the 16 pre-registered rows** (frozen §0.7 list, amended
2026-07-19/20/21; `C1_aug_s43` dropped 2026-07-22, stress-test L6). Every access
goes through the logging harness, so each execution appends to
`test_invocations.jsonl` — re-runs are visible forever.

**Two modes, `SET` below:**
- `SET = "val"` (**default — a DRY RUN**): exercises the entire 16-row path on
  the val sets. The harness logs a test invocation ONLY on `set_name=="test"`
  (verified: `harness.py` lines 354/440/512), so a val run writes **no**
  `test_invocations.jsonl` entry and is free to re-run. If the dry run completes
  clean and the per-row summary is sane, the test session will too. This is the
  de-risk step — run it first.
- `SET = "test"`: the real, once-only session. Requires flipping `SET` **and**
  setting `I_CONFIRM_SINGLE_TEST_SESSION = True`.

Nothing here selects a checkpoint or changes a split — selection happened on
val, upstream. This notebook only reads frozen artifacts and evaluates.

## Environment setup (repo + Drive + staging)

Staging is needed: `evaluate`/`cache_features` run the encoder over real data. All outputs (CSVs, feature caches, the test-invocation log) go to a local `SESSION_DIR`, so the run needs only **read** access to the Drive checkpoints — it never writes to the run folders.

In [1]:
from pathlib import Path
import subprocess
import sys
import zipfile

import numpy as np
import torch

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"

cwd = Path.cwd().resolve()
if (cwd / "sharp_har").exists():
    REPO_DIR = cwd
elif (cwd.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent
else:
    REPO_DIR = Path("/content/sharp-har")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

get_ipython().system("pip install -q -r {}/requirements.txt".format(REPO_DIR))

sys.path.insert(0, str(REPO_DIR))

from google.colab import drive
from sharp_har.utils import read_yaml

paths_cfg = read_yaml(REPO_DIR / "configs" / "paths.yaml")
drive_root = Path(paths_cfg["drive_root"])
stage_dir = Path(paths_cfg["stage_dir"])
CKPT_ROOT = Path(paths_cfg["ckpt_root"])

drive.mount("/content/drive")

if not (stage_dir.exists() and any(stage_dir.rglob("*.txt"))):
    stage_dir.mkdir(parents=True, exist_ok=True)
    for zip_name in paths_cfg["zips"]:
        src = drive_root / zip_name
        dst = Path("/content") / zip_name
        print("copying", src, "->", dst)
        subprocess.run(["cp", str(src), str(dst)], check=True)
        with zipfile.ZipFile(dst) as zf:
            zf.extractall(stage_dir)

print("Repo:", REPO_DIR, "| GPU:", torch.cuda.is_available(), "| CKPT_ROOT:", CKPT_ROOT)

Mounted at /content/drive
copying /content/drive/MyDrive/DATASET_SHARP/doppler_traces.zip -> /content/doppler_traces.zip
copying /content/drive/MyDrive/DATASET_SHARP/doppler_traces_S4_S5.zip -> /content/doppler_traces_S4_S5.zip
Repo: /content/sharp-har | GPU: True | CKPT_ROOT: /content/drive/MyDrive/sharp_har_runs


## Mode selector — DEFAULT IS THE VAL DRY RUN

Flip to `"test"` only for the real, once-only session, and only after a clean dry run.

In [4]:
SET = "test"   # "val" = dry run (no test contact) | "test" = the single §0.7 session
I_CONFIRM_SINGLE_TEST_SESSION = True   # must be True to run SET="test"

assert SET in ("val", "test")
if SET == "test":
    assert I_CONFIRM_SINGLE_TEST_SESSION, (
        "SET='test' is the ONE-TIME §0.7 session. Set I_CONFIRM_SINGLE_TEST_SESSION=True "
        "to proceed — after a clean val dry run. Every row will append to test_invocations.jsonl."
    )
    print("*** REAL TEST SESSION — this evaluates the held-out test sets ONCE ***")
else:
    print("DRY RUN on val — no test contact, no test_invocations.jsonl entry, safe to re-run.")

# Per-row output staging (kept off the Drive run folders so a re-run is clean).
SESSION_DIR = Path("/content/session_out") / SET
SESSION_DIR.mkdir(parents=True, exist_ok=True)
P2_LAB = REPO_DIR / "splits" / "p2_lab.json"
P2_LIVING = REPO_DIR / "splits" / "p2_living.json"
P1_SHARP = REPO_DIR / "splits" / "p1_sharp.json"

*** REAL TEST SESSION — this evaluates the held-out test sets ONCE ***


## The frozen 16-row list + readiness assert

The rows are declared once, here — the §0.7 hard-coded list. The assert checks
every required artifact (checkpoint / probe head / phase-B selection) exists on
Drive **before any evaluation runs**; a missing one stops the session.

Row kinds:
- `e2e` — end-to-end CE checkpoint via `evaluate` (optional `adapt_bn`).
- `c0` — `evaluate_c0` (paper decision fusion, P1, 5-class).
- `probe` — cache features then `evaluate_features` with a persisted linear head.
- `t3a` — cache features (raw or post-AdaBN) then a T3A prototype head.

The aug arm's seed twin `C1_aug_s43` was dropped (stress-test L6, ratified
2026-07-22, CHANGELOG): the paired design already controls the init, and the s43
twin re-uses the S7 test set, so it adds no independent replication over `C1_aug`
(the cross-rotation `C1_s6out_aug` twin is the one kept). The list is 16 rows;
the assert adapts to whatever is declared here.

In [5]:
# key -> row spec. folder = Drive run folder under CKPT_ROOT.
ROWS = [
    dict(key="C0",            kind="c0",    folder="C0",            split=P1_SHARP),
    dict(key="C1",            kind="e2e",   folder="C1",            split=P2_LAB),
    dict(key="C1_s43",        kind="e2e",   folder="C1_s43",        split=P2_LAB),
    dict(key="C2",            kind="e2e",   folder="C2",            split=P2_LAB),
    dict(key="C2_s43",        kind="e2e",   folder="C2_s43",        split=P2_LAB),
    dict(key="C1_lin",        kind="probe", folder="C1", ckpt="best.ckpt",   head="probe_head_best.npz",    split=P2_LAB),
    dict(key="C2_lin",        kind="probe", folder="C2", ckpt="best.ckpt",   head="probe_head_best.npz",    split=P2_LAB),
    dict(key="C3",            kind="probe", folder="C3", from_phaseb=True,                                  split=P2_LAB),
    dict(key="C3_ft",         kind="e2e",   folder="C3_ft",         split=P2_LAB),
    dict(key="C1_AdaBN",      kind="e2e",   folder="C1", adapt_bn=True,       split=P2_LAB),
    dict(key="C1_T3A",        kind="t3a",   folder="C1", adapt_bn=False,      split=P2_LAB),
    dict(key="C1_AdaBN_T3A",  kind="t3a",   folder="C1", adapt_bn=True,       split=P2_LAB),
    dict(key="C1_s6out",      kind="e2e",   folder="C1_s6out",      split=P2_LIVING),
    dict(key="C1_aug",        kind="e2e",   folder="C1_aug",        split=P2_LAB),
    dict(key="C1_s6out_aug",  kind="e2e",   folder="C1_s6out_aug",  split=P2_LIVING),
    dict(key="C1_sharplike",  kind="e2e",   folder="C1_sharplike",  split=P2_LAB),
]
print(f"{len(ROWS)} rows declared (frozen §0.7 list)")

# Readiness assert (§0.7): sharp_har.session.required_paths knows what each row
# kind needs on Drive (incl. rebuilding the C3 epoch/head from selected_epoch,
# since the stored absolute paths may not resolve under a different mount).
# A missing artifact stops the session before any evaluation runs.
from sharp_har.session import readiness_missing

missing = readiness_missing(ROWS, CKPT_ROOT)
if missing:
    print("READINESS FAILED — missing artifacts:")
    for k, p in missing:
        print(f"  [{k}] {p}")
    raise SystemExit("Not ready: every row needs its val-selected artifact on Drive "
                     "(add Editor shortcuts to EVERY run folder from this one account).")
print("READINESS OK — all", len(ROWS), "rows have their artifacts.")

16 rows declared (frozen §0.7 list)
READINESS OK — all 16 rows have their artifacts.


## Run the 16 rows

Each row writes windows/metrics/confusion CSVs into its own `SESSION_DIR/<key>/`; the summary reads the fused accuracy back for a sanity pass (this is what makes the dry run self-verifying).

In [6]:
from sharp_har.session import run_row, row_accuracy

# Dry run keeps its previews off the committable report dir; the real session
# writes the notebook-06 CSVs into reports/final/ for the §0.7 commit.
FINAL_DIR = (REPO_DIR / "reports" / "final") if SET == "test" else (SESSION_DIR / "_final_preview")

summary = []
for r in ROWS:
    res = run_row(r, SET, session_dir=SESSION_DIR, ckpt_root=CKPT_ROOT,
                  stage_dir=stage_dir, repo_dir=REPO_DIR, final_dir=FINAL_DIR)
    acc, ntr, nw = row_accuracy(res["out_dir"])
    summary.append((r["key"], acc, ntr, nw))
    print(f"  {r['key']:16s} acc {acc:.4f}  ({ntr} traces, {nw} fused windows)")
    if res["t3a_audit"] is not None:  # §9: surface the T3A audit (harness ignores it)
        a = res["t3a_audit"]
        print(f"  {'':16s} T3A supports/class {a['n_supports'].tolist()} "
              f"| pseudo-label counts {a['pseudo_label_counts'].tolist()}")
print("\nall rows done on", SET)

2026-07-22 15:05:21,526 [WARNING] sharp_har.harness: TEST-SET ACCESS logged to /content/session_out/test/C0/test_invocations.jsonl: {'utc': '2026-07-22T15:05:21+00:00', 'kind': 'evaluate', 'checkpoint': '/content/drive/MyDrive/sharp_har_runs/C0/best.ckpt', 'split_file': '/content/sharp-har/splits/p1_sharp.json', 'set_name': 'test', 'fusion': 'sharp_c0', 'adapt_bn': False, 'config_name': 'C0', 'git_hash': 'bdf52babb08bb127be02e9f622497bba5eb8a4f8'}
2026-07-22 15:05:21,849 [INFO] sharp_har.data: test: excluding 17 trace(s) with activities outside the class list ['E', 'J', 'L', 'R', 'W']: ['S3a_H', 'S3a_S', 'S4a_C1', 'S4a_C2', 'S4a_H1', 'S4a_H2', 'S4a_S', 'S5a_C1', 'S5a_C2', 'S5a_H1', 'S5a_H2', 'S6a_C', 'S6a_H', 'S6a_S', 'S7a_C', 'S7a_H', 'S7a_S']
2026-07-22 15:05:21,885 [INFO] sharp_har.data: test: 57 traces, 10868 (window, antenna) samples (win=340, stride=340)
2026-07-22 15:05:29,209 [INFO] sharp_har.harness: best_test_sharp_c0 [test, fused_sharp_c0]: accuracy 0.6117, macro-F1 0.6053 (

  C0               acc 0.6117  (57 traces, 2717 fused windows)


2026-07-22 15:05:33,201 [WARNING] sharp_har.harness: TEST-SET ACCESS logged to /content/session_out/test/C1/test_invocations.jsonl: {'utc': '2026-07-22T15:05:33+00:00', 'kind': 'evaluate', 'checkpoint': '/content/drive/MyDrive/sharp_har_runs/C1/best.ckpt', 'split_file': '/content/sharp-har/splits/p2_lab.json', 'set_name': 'test', 'fusion': 'softmax_avg', 'adapt_bn': False, 'config_name': 'C1', 'git_hash': 'bdf52babb08bb127be02e9f622497bba5eb8a4f8'}
2026-07-22 15:05:33,225 [INFO] sharp_har.data: test: 11 traces, 1700 (window, antenna) samples (win=340, stride=340)
2026-07-22 15:05:34,945 [INFO] sharp_har.harness: best_test_softmax_avg [test, fused_softmax_avg]: accuracy 0.8047, macro-F1 0.8038 (425 windows) -> /content/session_out/test/C1/best_test_softmax_avg_metrics.csv


  C1               acc 0.8047  (11 traces, 425 fused windows)


2026-07-22 15:05:38,021 [WARNING] sharp_har.harness: TEST-SET ACCESS logged to /content/session_out/test/C1_s43/test_invocations.jsonl: {'utc': '2026-07-22T15:05:38+00:00', 'kind': 'evaluate', 'checkpoint': '/content/drive/MyDrive/sharp_har_runs/C1_s43/best.ckpt', 'split_file': '/content/sharp-har/splits/p2_lab.json', 'set_name': 'test', 'fusion': 'softmax_avg', 'adapt_bn': False, 'config_name': 'C1_s43', 'git_hash': 'bdf52babb08bb127be02e9f622497bba5eb8a4f8'}
2026-07-22 15:05:38,042 [INFO] sharp_har.data: test: 11 traces, 1700 (window, antenna) samples (win=340, stride=340)
2026-07-22 15:05:39,521 [INFO] sharp_har.harness: best_test_softmax_avg [test, fused_softmax_avg]: accuracy 0.8000, macro-F1 0.7990 (425 windows) -> /content/session_out/test/C1_s43/best_test_softmax_avg_metrics.csv


...

  C1_aug           acc 0.7765  (11 traces, 425 fused windows)


2026-07-22 15:06:24,014 [WARNING] sharp_har.harness: TEST-SET ACCESS logged to /content/session_out/test/C1_s6out_aug/test_invocations.jsonl: {'utc': '2026-07-22T15:06:24+00:00', 'kind': 'evaluate', 'checkpoint': '/content/drive/MyDrive/sharp_har_runs/C1_s6out_aug/best.ckpt', 'split_file': '/content/sharp-har/splits/p2_living.json', 'set_name': 'test', 'fusion': 'softmax_avg', 'adapt_bn': False, 'config_name': 'C1_s6out_aug', 'git_hash': 'bdf52babb08bb127be02e9f622497bba5eb8a4f8'}
2026-07-22 15:06:24,041 [INFO] sharp_har.data: test: 15 traces, 2864 (window, antenna) samples (win=340, stride=340)
2026-07-22 15:06:26,217 [INFO] sharp_har.harness: best_test_softmax_avg [test, fused_softmax_avg]: accuracy 0.8156, macro-F1 0.8227 (716 windows) -> /content/session_out/test/C1_s6out_aug/best_test_softmax_avg_metrics.csv


  C1_s6out_aug     acc 0.8156  (15 traces, 716 fused windows)


2026-07-22 15:06:26,766 [WARNING] sharp_har.harness: TEST-SET ACCESS logged to /content/session_out/test/C1_sharplike/test_invocations.jsonl: {'utc': '2026-07-22T15:06:26+00:00', 'kind': 'evaluate', 'checkpoint': '/content/drive/MyDrive/sharp_har_runs/C1_sharplike/best.ckpt', 'split_file': '/content/sharp-har/splits/p2_lab.json', 'set_name': 'test', 'fusion': 'softmax_avg', 'adapt_bn': False, 'config_name': 'C1_sharplike', 'git_hash': 'bdf52babb08bb127be02e9f622497bba5eb8a4f8'}
2026-07-22 15:06:26,782 [INFO] sharp_har.data: test: 11 traces, 1700 (window, antenna) samples (win=340, stride=340)
2026-07-22 15:06:27,463 [INFO] sharp_har.harness: best_test_softmax_avg [test, fused_softmax_avg]: accuracy 0.5694, macro-F1 0.5434 (425 windows) -> /content/session_out/test/C1_sharplike/best_test_softmax_avg_metrics.csv


  C1_sharplike     acc 0.5694  (11 traces, 425 fused windows)

all rows done on test


## Summary table

Sanity pass. On the dry run: numbers near the val macro-F1s already on record (accuracy, not macro-F1, so not identical) and no crash = the path is sound. On the real session: this is the headline table (read alongside notebook 06's paired bootstrap + per-class decomposition).

In [7]:
import pandas as pd
tab = pd.DataFrame(summary, columns=["row", "accuracy", "n_traces", "n_windows"])
print(tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\nSET = {SET} | CSVs in {FINAL_DIR} named <row>_{SET}_<fusion>_<kind>.csv (notebook 06 contract)")

         row  accuracy  n_traces  n_windows
          C0    0.6117        57       2717
          C1    0.8047        11        425
      C1_s43    0.8000        11        425
          C2    0.6471        11        425
      C2_s43    0.6000        11        425
      C1_lin    0.8071        11        425
      C2_lin    0.7200        11        425
          C3    0.7271        11        425
       C3_ft    0.7200        11        425
    C1_AdaBN    0.7459        11        425
      C1_T3A    0.8118        11        425
C1_AdaBN_T3A    0.7576        11        425
    C1_s6out    0.7430        15        716
      C1_aug    0.7765        11        425
C1_s6out_aug    0.8156        15        716
C1_sharplike    0.5694        11        425

SET = test | CSVs in /content/sharp-har/reports/final named <row>_test_<fusion>_<kind>.csv (notebook 06 contract)


## Commit the measured artifacts (real session only)

On `SET="test"`, commit **in one commit**: `reports/final/` (the per-row CSVs +
each run folder's `test_invocations.jsonl`), this executed notebook under
`notebooks/runs/`, and the `STATUS.md` update. Then run notebook 06 on
`reports/final/` for the paired bootstrap, calibration and per-class /
class-coverage decomposition (§9 report material).

In [8]:
if SET == "test":
    final = REPO_DIR / "reports" / "final"
    logs = list(SESSION_DIR.glob("*/test_invocations.jsonl"))
    with open(final / "test_invocations.jsonl", "w", encoding="utf-8") as agg:
        for lg in sorted(logs):
            agg.write(lg.read_text(encoding="utf-8"))
    print(f"merged {len(logs)} per-row test-invocation logs into", final / "test_invocations.jsonl")
    print("now: git add reports/final + this notebook (notebooks/runs/) + STATUS, one commit.")
else:
    print("dry run — nothing to commit. Flip SET to 'test' (and confirm) for the real session.")

merged 16 per-row test-invocation logs into /content/sharp-har/reports/final/test_invocations.jsonl
now: git add reports/final + this notebook (notebooks/runs/) + STATUS, one commit.
